In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")

load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [2]:
previous_application_df.head(5)

,id_prev,id_curr,name_contract_type,amt_annuity,amt_application,amt_credit,amt_down_payment,amt_down_payment_is_missing,amt_goods_price,amt_goods_price_is_missing,...,days_first_due_has_sentinel_value,days_first_due,days_last_due_1st_version_has_sentinel_value,days_last_due_1st_version,days_last_due_has_sentinel_value,days_last_due,days_ termination_has_sentinel_value,days_termination,nflag_insured_on_approval,days_and_insurance_information_are_missing
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,0,17145.0,0,...,0,-42.0,0,300.0,0,-42.0,0,-37.0,0.0,0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,1,607500.0,0,...,0,-134.0,0,916.0,1,NaN,1,NaN,1.0,0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,1,112500.0,0,...,0,-271.0,0,59.0,1,NaN,1,NaN,1.0,0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,1,450000.0,0,...,0,-482.0,0,-152.0,0,-182.0,0,-177.0,1.0,0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,1,337500.0,0,...,0,NaN,0,NaN,0,NaN,0,NaN,NaN,1


In [ ]:
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_df= previous_application_df.drop(columns="id_prev")
previous_application_df= previous_application_df.loc[~mask_no_final]
previous_application_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_df.groupby("id_curr").head(3)


In [8]:
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
dtale.show(df_wide)

In [ ]:

previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])


previous_application_df.groupby("id_curr").agg({

    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],

    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"],

    #others_monetary
    "diff_application_credit": ["max","mean","min","median","sum"],
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","std","min","median"],

    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","min","max","sum"],
})

